In [ ]:
# ============================================
# SIMPLE ORTHODONTIC SUITABILITY CHECK
# ============================================

import torch
import numpy as np
import nibabel as nib
from scipy.ndimage import zoom

def check_orthodontic_suitability(cbct_path, model_path='/content/drive/MyDrive/MMDental/models/best_model.pth'):
    """
    Simple check if patient is suitable for orthodontic treatment
    Rule: ANY disease with probability > 20% = NOT suitable
    """

    # Load model
    checkpoint = torch.load(model_path, map_location='cpu', weights_only=False)

    class Simple3DCNN(torch.nn.Module):
        def __init__(self, num_classes=8):
            super().__init__()
            self.conv_layers = torch.nn.Sequential(
                torch.nn.Conv3d(1, 32, kernel_size=3, padding=1),
                torch.nn.BatchNorm3d(32),
                torch.nn.ReLU(inplace=True),
                torch.nn.MaxPool3d(2),
                torch.nn.Conv3d(32, 64, kernel_size=3, padding=1),
                torch.nn.BatchNorm3d(64),
                torch.nn.ReLU(inplace=True),
                torch.nn.MaxPool3d(2),
                torch.nn.Conv3d(64, 128, kernel_size=3, padding=1),
                torch.nn.BatchNorm3d(128),
                torch.nn.ReLU(inplace=True),
                torch.nn.MaxPool3d(2),
                torch.nn.Conv3d(128, 256, kernel_size=3, padding=1),
                torch.nn.BatchNorm3d(256),
                torch.nn.ReLU(inplace=True),
                torch.nn.AdaptiveAvgPool3d((1, 1, 1))
            )
            self.classifier = torch.nn.Sequential(
                torch.nn.Dropout(0.5),
                torch.nn.Linear(256, 128),
                torch.nn.ReLU(inplace=True),
                torch.nn.Dropout(0.3),
                torch.nn.Linear(128, num_classes)
            )

        def forward(self, x):
            x = self.conv_layers(x)
            x = x.view(x.size(0), -1)
            return self.classifier(x)

    model = Simple3DCNN(num_classes=len(checkpoint['class_names']))
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()

    # Load and preprocess CBCT
    nifti = nib.load(cbct_path)
    volume = nifti.get_fdata().astype(np.float32)
    volume = (volume - volume.min()) / (volume.max() - volume.min() + 1e-8)

    factors = (64/volume.shape[0], 64/volume.shape[1], 64/volume.shape[2])
    volume = zoom(volume, factors, order=1)
    volume = np.expand_dims(volume, axis=(0, 1))
    volume = torch.from_numpy(volume).float()

    # Predict
    with torch.no_grad():
        outputs = model(volume)
        probs = torch.nn.functional.softmax(outputs, dim=1)

    probs = probs.numpy()[0]
    class_names = checkpoint['class_names']

    # Check for any disease above 20%
    diseases_above_threshold = []
    for i, (condition, prob) in enumerate(zip(class_names, probs)):
        if prob > 0.20:
            diseases_above_threshold.append((condition, prob))

    # Print results
    print("\n" + "="*60)
    print("ORTHODONTIC SUITABILITY CHECK")
    print("="*60)

    print("\n📊 DISEASE DETECTION RESULTS:")
    print("-" * 40)
    for condition, prob in zip(class_names, probs):
        marker = "⚠️" if prob > 0.20 else "  "
        print(f"{marker} {condition:20s}: {prob:.1%}")

    print("\n" + "="*60)

    # Decision
    if len(diseases_above_threshold) == 0:
        print("\n✅ VERDICT: SUITABLE FOR ORTHODONTIC TREATMENT")
        print("   No significant dental diseases detected (>20% threshold)")
        print("   Action: Proceed with orthodontic treatment planning")
        suitable = True
    else:
        print("\n❌ VERDICT: NOT SUITABLE FOR ORTHODONTIC TREATMENT")
        print(f"   Diseases detected above 20% threshold:")
        for disease, prob in diseases_above_threshold:
            print(f"   - {disease}: {prob:.1%}")
        print("\n   Action: Complete dental treatment before orthodontics")
        suitable = False

    print("\n" + "="*60)

    return {
        'suitable': suitable,
        'diseases': diseases_above_threshold,
        'all_probabilities': dict(zip(class_names, probs))
    }


# CHANGE THIS PATH TO YOUR CBCT FILE
cbct_file_path = '/content/drive/MyDrive/MMDental/MMDental/12/12.nii.gz'

# Run the check
result = check_orthodontic_suitability(cbct_file_path)

# Access results programmatically if needed
if result['suitable']:
    print("\n💡 Patient can start orthodontic treatment")
else:
    print(f"\n💡 Treat these first: {', '.join([d[0] for d in result['diseases']])}")